# Planning-Agent Evaluation (Refined) — Deterministic PlanScore Ablation

A reproducible ablation of the **retrieval context** supplied to the RepurAgent
planning agent, scored with a **deterministic, judge-free** metric.

This notebook deliberately fuses the strong points of two earlier notebooks:

**Inherited from `ablation_study.ipynb`**
- For each configuration we *always* call the real `protocol_search_sop` /
  `literature_search_pubmed` tools and feed the retrieved text into the planner's
  context (deterministic, cached context injection — the arm decides what context
  exists, not the model's tool-calling whims).
- Nonparametric statistics (Kruskal–Wallis → pairwise Mann–Whitney U → Holm).
- Publication-style plots (mean ± 95% CI bars, pairwise significance heatmap).

**Inherited from `plan_evaluation.ipynb`**
- The planner is built from the **core components** — it reuses the production
  planning prompt (`core.prompts.prompts.PLANNING_SYSTEM_PROMPT_ver3`) and the real
  research tools, rather than a re-implemented prompt.
- A **deterministic** plan score, independent of any LLM judge:

  $$\text{PlanScore} = \text{OrderPreservation} \times \text{ContentF1}$$

  Generated steps are matched to ground-truth steps by Hungarian assignment over
  OpenAI-embedding cosine similarity; the similarity threshold τ is calibrated
  against paraphrase / non-paraphrase pairs (Youden's *J*).

**Ablation design** — a full `2³` factorial over the three context sources fed to
the planner, giving **8 configurations** labelled `S{sop}-L{lit}-E{episodic}`:

| source          | flag | retrieval |
|-----------------|:----:|-----------|
| SOP / protocol  | `S`  | `protocol_search_sop` |
| literature      | `L`  | `literature_search_pubmed` |
| episodic memory | `E`  | `orchestrator.get_episodic_context` (past planning examples) |

e.g. `S0-L0-E0` = vanilla (no context), `S1-L1-E1` = all three sources.

**Ground truth** is the `reference_steps` workflow shipped with each request in
`planning_requests.csv`.

## 1. Setup, imports & core components

In [11]:
import sys
import json
import time
import ast
import warnings
from itertools import combinations, product
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.optimize import linear_sum_assignment
from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import os
warnings.filterwarnings("ignore")

# --- Locate the project root (dir containing both app/ and backend/) ---------
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "app").is_dir() and (PROJECT_ROOT / "backend").is_dir()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if not ((PROJECT_ROOT / "app").is_dir() and (PROJECT_ROOT / "backend").is_dir()):
    raise RuntimeError(f"Could not locate project root from {CURRENT_DIR}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Core components imported from the application (NOT re-implemented) ------
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent

from app.config import OPENAI_API_KEY
from core.prompts.prompts import PLANNING_SYSTEM_PROMPT_ver3          # production planner prompt
from backend.utils.research_tools import (                            # production research tools
    literature_search_pubmed,
    protocol_search_sop,
)
from persistence.memory.episodic_memory.episodic_learning import get_orchestrator  # episodic memory

from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

ABLATION_DIR = PROJECT_ROOT / "analysis" / "Ablation"
OUTPUT_DIR = ABLATION_DIR / "outputs" / "planning_eval_refine"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("project root :", PROJECT_ROOT)
print("output dir   :", OUTPUT_DIR)

project root : /Users/dinhu955/Desktop/RepurAgent/repuragent-web
output dir   : /Users/dinhu955/Desktop/RepurAgent/repuragent-web/analysis/Ablation/outputs/planning_eval_refine


In [15]:
# ------------------------- publication plotting theme ----------------------- #
COLORS = ['#4477AA', '#EE6677', '#228833', '#CCBB44', '#66CCEE', '#AA3377', '#BBBBBB']
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.titlesize': 9,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'lines.linewidth': 1.0,
    'patch.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
})
sns.set_theme(style="ticks")
pd.set_option("display.max_colwidth", 160)

## 2. Ablation configurations & benchmark tasks

In [16]:
# Full 2^3 factorial over the three context sources fed to the planner.
CONFIGS = {
    f"S{s}-L{l}-E{e}": {"use_sop": s, "use_lit": l, "use_episodic": e}
    for s, l, e in product([0, 1], repeat=3)
}
CONFIG_ORDER = list(CONFIGS.keys())                 # S0-L0-E0 ... S1-L1-E1
CONFIG_LABELS = {c: c for c in CONFIG_ORDER}        # factorial code is the label
_pal = sns.color_palette("colorblind", n_colors=len(CONFIG_ORDER))
CONFIG_COLORS = dict(zip(CONFIG_ORDER, _pal))
print(f"{len(CONFIG_ORDER)} configs:", CONFIG_ORDER)

# Benchmark requests + ground-truth reference workflows.
TASKS_df = pd.read_csv(ABLATION_DIR / "planning_requests.csv")
TASKS = []
for rec in TASKS_df.to_dict(orient="records"):
    ref = rec["reference_steps"]
    rec["reference_steps"] = ast.literal_eval(ref) if isinstance(ref, str) else list(ref)
    TASKS.append(rec)

print(f"{len(TASKS)} benchmark tasks")
print("example ground-truth steps:", TASKS[0]["reference_steps"][:2])

8 configs: ['S0-L0-E0', 'S0-L0-E1', 'S0-L1-E0', 'S0-L1-E1', 'S1-L0-E0', 'S1-L0-E1', 'S1-L1-E0', 'S1-L1-E1']
15 benchmark tasks
example ground-truth steps: ['Knowledge Graph Creation. Querying gene, pathways, mechanism of action. Get Drugs for these entities', 'Data Merging and Filtering']


## 3. Deterministic context retrieval (per config)

Following the ablation notebook, we *always* call the real research tools /
episodic-memory orchestrator for the relevant arm and inject the retrieved text
into the planner's context. Results are cached per task so every repeat and every
arm sees identical retrieved text — the only thing that varies across arms is
**which** context sources are present.

In [17]:
orchestrator = get_orchestrator()

SOP_CACHE, LIT_CACHE, EPISODIC_CACHE = {}, {}, {}
SOP_MAX_CHARS, LIT_MAX_CHARS, LIT_LIMIT = 3500, 3000, 4
EPISODIC_MAX_EXAMPLES = 2


def _truncate(text: str, max_chars: int) -> str:
    text = (text or "").strip()
    return text if len(text) <= max_chars else text[:max_chars] + "\n...[truncated]"


def get_sop_context(task: dict) -> str:
    tid = task["task_id"]
    if tid not in SOP_CACHE:
        text = protocol_search_sop.invoke({"query": task["protocol_query"]})
        SOP_CACHE[tid] = _truncate(text, SOP_MAX_CHARS)
    return SOP_CACHE[tid]


def get_lit_context(task: dict) -> str:
    tid = task["task_id"]
    if tid not in LIT_CACHE:
        text = literature_search_pubmed.invoke({"query": task["literature_query"], "limit": LIT_LIMIT})
        LIT_CACHE[tid] = _truncate(text, LIT_MAX_CHARS)
    return LIT_CACHE[tid]


def get_episodic_examples(task: dict) -> list[dict]:
    tid = task["task_id"]
    if tid not in EPISODIC_CACHE:
        ctx = orchestrator.get_episodic_context(task["request"])
        EPISODIC_CACHE[tid] = ctx.get("examples", [])[:EPISODIC_MAX_EXAMPLES]
    return EPISODIC_CACHE[tid]


def format_episodic_examples(examples: list[dict]) -> str:
    if not examples:
        return ""
    blocks = []
    for ex in examples:
        decomposition = ex.get("final_decomposition") or ex.get("initial_decomposition") or ""
        blocks.append(
            f"Input: {ex.get('task', '')}\n```\n"
            f"📋 BREAKDOWN: {decomposition}\n\n"
            f"📋 Note for success: {ex.get('notes', '')}\n```"
        )
    return "Relevant past planning examples:\n" + "\n\n".join(blocks)


def build_context(task: dict, use_sop: int, use_lit: int, use_episodic: int):
    """Assemble the retrieved-context block injected into the planner message.

    Returns (context_text, n_episodic_examples_used).
    """
    sections = []
    if use_sop:
        sections.append("## SOP / protocol context retrieved for planning:\n" + get_sop_context(task))
    if use_lit:
        sections.append("## Scientific literature context retrieved for planning:\n" + get_lit_context(task))
    n_episodic = 0
    if use_episodic:
        examples = get_episodic_examples(task)
        n_episodic = len(examples)
        episodic_block = format_episodic_examples(examples)
        if episodic_block:
            sections.append("## " + episodic_block)
    return "\n\n".join(sections), n_episodic

## 4. Planning agent from core components

The planner reuses the **production planning prompt** and is instantiated with the
same `create_react_agent` machinery the app uses (`core.agents.planning_agent`).
For a clean, deterministic ablation the tools are *not* wired into the agent —
instead each arm's context is injected directly into the request. This guarantees
each arm's context is exactly what the config specifies (rather than depending on
whether/what the model chooses to search), while the plan itself is still produced
by the real production prompt in its canonical `BREAKDOWN` format.

In [18]:
PLANNER_MODEL_NAME = "gpt-4o"
planner_llm = init_chat_model(
    PLANNER_MODEL_NAME, model_provider="openai", api_key=OPENAI_API_KEY, temperature=0,
)

# Eval-scoped planner: production prompt, no wired tools (context is injected).
planning_agent = create_react_agent(
    model=planner_llm,
    tools=[],
    name="planning_agent",
    prompt=PLANNING_SYSTEM_PROMPT_ver3,
)


def run_planning_agent(task: dict, context: str) -> str:
    """Run the planner on one request with pre-retrieved context; return raw plan text."""
    if context.strip():
        message = (
            f"{task['request']}\n\n"
            "# RETRIEVED CONTEXT (provided for this planning session; "
            "external tool calls are disabled for this evaluation)\n"
            f"{context}"
        )
    else:
        message = task["request"]

    result = planning_agent.invoke({"messages": message}, config={"recursion_limit": 10})
    for m in reversed(result["messages"]):
        if getattr(m, "type", "") != "ai":
            continue
        content = getattr(m, "content", "")
        if isinstance(content, list):
            content = "".join(
                p.get("text", "") if isinstance(p, dict) else str(p) for p in content
            )
        if content and content.strip():
            return content
    return ""

## 5. Plan parser (structured output)

The planner emits a free-text `📋 BREAKDOWN` plan. A small structured-output model
condenses it into an ordered list of concise action phrases, joined with ` → ` —
the same normalized representation used for the ground-truth `reference_steps`.

In [19]:
from pydantic import BaseModel, Field

PARSER_MODEL_NAME = "gpt-4o-mini"


class FinalPlan(BaseModel):
    """Condensed, ordered plan extracted from the planning agent's canonical output."""
    steps: list[str] = Field(
        description=(
            "Ordered list of concise action phrases (roughly 3-8 words each), one per "
            "atomic plan step, in execution order. Strip numbering, the "
            "'Note for success' line, and any approval prompt."
        )
    )


PARSER_SYSTEM = (
    "You convert a planning agent's canonical BREAKDOWN plan into a concise ordered "
    "step list.\n"
    "For each plan step emit exactly one short action phrase (about 3-8 words) "
    "capturing that step's action and output.\n"
    "- Preserve the original order.\n"
    "- Drop numbering/arrows, the 'Note for success' line, and any 'Please review'/"
    "approval prompt.\n"
    "- Use imperative verb phrases (e.g. 'build knowledge graph', 'run ADMET prediction').\n"
    "- Do not invent, merge, or add commentary."
)

parser_llm = init_chat_model(
    PARSER_MODEL_NAME, model_provider="openai", api_key=OPENAI_API_KEY, temperature=0,
).with_structured_output(FinalPlan)


def parse_plan(plan_text: str) -> str:
    """Canonical plan text -> ' → '-joined concise step string."""
    if not plan_text or not plan_text.strip():
        return ""
    parsed = parser_llm.invoke([("system", PARSER_SYSTEM), ("human", plan_text)])
    return " → ".join(s.strip() for s in parsed.steps if s and s.strip())

## 6. Generate plans — N repeats × tasks × configs

For each repeat, task and config we (1) retrieve & inject the arm's context,
(2) run the core planner, (3) parse to a normalized step string. The ground-truth
plan is the ` → `-joined `reference_steps`. One tidy long-format row per plan.

In [ ]:
N_REPEATS = 5
gen_rows = []

for run_idx in range(1, N_REPEATS + 1):
    print(f"\n=== Repeat {run_idx}/{N_REPEATS} ===")
    for task in TASKS:
        gt_plan = " → ".join(task["reference_steps"])
        for config_name in CONFIG_ORDER:
            cfg = CONFIGS[config_name]
            context, n_episodic = build_context(task, cfg["use_sop"], cfg["use_lit"], cfg["use_episodic"])
            try:
                raw_plan = run_planning_agent(task, context)
            except Exception as exc:
                print(f"  retry {task['task_id']} | {config_name}: {exc}")
                raw_plan = run_planning_agent(task, context)
            generated = parse_plan(raw_plan)
            gen_rows.append({
                "run": run_idx,
                "task_id": task["task_id"],
                "config": config_name,
                "use_sop": cfg["use_sop"],
                "use_lit": cfg["use_lit"],
                "use_episodic": cfg["use_episodic"],
                "query": task["request"],
                "ground_truth": gt_plan,
                "generated": generated,
                "episodic_examples_used": n_episodic,
                "raw_plan": raw_plan,
            })
            print(f"  {task['task_id']:<48} | {config_name:<8} | {len(generated.split('→'))} steps")
        time.sleep(0.5)

long_df = pd.DataFrame(gen_rows)
long_df.to_csv(OUTPUT_DIR / "planning_eval_generated.csv", index=False)
print(f"\nGenerated {len(long_df)} plans")
long_df.head()

## 7. Deterministic scoring machinery (PlanScore = OP × F1)

Steps are embedded with OpenAI `text-embedding-3-large`. Generated vs ground-truth
steps are matched by Hungarian assignment over cosine similarity. A match counts as
valid when similarity ≥ τ.

- **Content F1** — harmonic mean of precision (valid / #generated) and recall
  (valid / #ground-truth).
- **Order Preservation** — length of the longest increasing subsequence of matched
  ground-truth indices (in generated order), divided by #valid matches.
- **PlanScore** = OrderPreservation × ContentF1.

In [20]:
from openai import OpenAI

EMBEDDING_MODEL = "text-embedding-3-large"
EMBED_BATCH = 2048
_oai = OpenAI(api_key=OPENAI_API_KEY)


def embed_texts(texts, model=EMBEDDING_MODEL, batch_size=EMBED_BATCH):
    """Embed and L2-normalize a list of texts using the OpenAI embeddings API."""
    all_emb = []
    n_batches = (len(texts) + batch_size - 1) // batch_size
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        print(f"  embed batch {i // batch_size + 1}/{n_batches}: {len(batch)} texts...", end=" ", flush=True)
        t0 = time.time()
        resp = _oai.embeddings.create(input=batch, model=model)
        all_emb.extend(d.embedding for d in resp.data)
        print(f"done ({time.time() - t0:.1f}s)")
    emb = np.asarray(all_emb, dtype=np.float32)
    norms = np.linalg.norm(emb, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return emb / norms


def split_steps(plan_str):
    """' → '-joined plan string -> list of step phrases."""
    if pd.isna(plan_str) or not str(plan_str).strip():
        return []
    return [s.strip() for s in str(plan_str).split("→") if s.strip()]


def lis_length(seq):
    """Longest increasing subsequence length (patience sorting, O(n log n))."""
    tails = []
    for x in seq:
        lo, hi = 0, len(tails)
        while lo < hi:
            mid = (lo + hi) // 2
            if tails[mid] < x:
                lo = mid + 1
            else:
                hi = mid
        if lo == len(tails):
            tails.append(x)
        else:
            tails[lo] = x
    return len(tails)


def compute_plan_score(gt_steps, gen_steps, sim_matrix, step_to_idx, tau):
    """PlanScore = ContentF1 × OrderPreservation for one (ground-truth, generated) pair."""
    if not gt_steps or not gen_steps:
        return {"content_f1": 0.0, "order_preservation": 0.0, "plan_score": 0.0,
                "n_matched": 0, "n_valid": 0, "precision": 0.0, "recall": 0.0}

    n_gt, n_gen = len(gt_steps), len(gen_steps)
    cost = np.ones((n_gt, n_gen))
    for i, gs in enumerate(gt_steps):
        gi = step_to_idx.get(gs)
        if gi is None:
            continue
        for j, gns in enumerate(gen_steps):
            gj = step_to_idx.get(gns)
            if gj is None:
                continue
            cost[i, j] = 1.0 - sim_matrix[gi, gj]

    row_ind, col_ind = linear_sum_assignment(cost)
    matched = []
    for r, c in zip(row_ind, col_ind):
        gi, gj = step_to_idx.get(gt_steps[r]), step_to_idx.get(gen_steps[c])
        if gi is not None and gj is not None:
            matched.append((r, c, sim_matrix[gi, gj]))

    valid = [(r, c, s) for r, c, s in matched if s >= tau]
    n_valid = len(valid)

    tp = n_valid
    precision = tp / n_gen if n_gen else 0.0
    recall = tp / n_gt if n_gt else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    if n_valid == 1:
        op = 1.0
    elif n_valid == 0:
        op = 0.0
    else:
        gt_pos_seq = [x[0] for x in sorted(valid, key=lambda x: x[1])]  # order by gen position
        op = lis_length(gt_pos_seq) / n_valid

    return {"content_f1": f1, "order_preservation": op, "plan_score": f1 * op,
            "n_matched": len(matched), "n_valid": n_valid,
            "precision": precision, "recall": recall}

## 8. Calibrate the similarity threshold τ

τ is chosen to best separate drug-repurposing **paraphrase** step pairs from
**non-paraphrase** pairs, maximizing Youden's *J* = sensitivity + specificity − 1.

In [21]:
PARAPHRASE_PAIRS = [
    ("build knowledge graph for disease", "construct disease knowledge graph"),
    ("query genes and pathways", "retrieve associated genes and pathways"),
    ("extract candidate drugs from KG", "get drugs for graph entities"),
    ("merge and filter datasets", "combine and filter candidate tables"),
    ("run ADMET prediction models", "predict ADMET properties"),
    ("predict repurposing indication", "score drug repurposing potential"),
    ("rank candidate drugs", "prioritize drug candidates"),
    ("generate final report", "compile workflow summary report"),
    ("standardize SMILES structures", "normalize chemical structures"),
    ("literature search for disease biology", "gather disease mechanism evidence"),
    ("search SOP for repurposing protocol", "retrieve governing SOP"),
    ("visualize ranked results", "plot prioritized candidates"),
    ("compute integrated score", "calculate composite ranking score"),
    ("filter by hERG liability", "remove cardiotoxic candidates"),
    ("predict blood-brain barrier permeability", "estimate BBB penetration"),
    ("identify drug targets", "find candidate protein targets"),
    ("inspect uploaded compound list", "review input compound dataset"),
    ("select IC50 assay concentrations", "propose testable assay concentrations"),
]

NON_PARAPHRASE_PAIRS = [
    ("build knowledge graph for disease", "select required PPE"),
    ("run ADMET prediction models", "generate final report"),
    ("query genes and pathways", "visualize ranked results"),
    ("standardize SMILES structures", "search literature for efficacy"),
    ("rank candidate drugs", "inspect uploaded compound list"),
    ("predict blood-brain barrier permeability", "merge and filter datasets"),
    ("filter by hERG liability", "construct disease knowledge graph"),
    ("extract candidate drugs from KG", "compute solubility"),
    ("literature search for disease biology", "prioritize drug candidates"),
    ("compile workflow summary report", "identify drug targets"),
    ("search SOP for repurposing protocol", "run in silico predictions"),
    ("select IC50 assay concentrations", "normalize chemical structures"),
    ("compute integrated score", "gather disease mechanism evidence"),
    ("estimate BBB penetration", "get drugs for graph entities"),
    ("visualize ranked results", "predict CYP inhibition"),
    ("review input compound dataset", "draft regulatory rationale"),
    ("plot prioritized candidates", "retrieve governing SOP"),
    ("predict ADMET properties", "query clinical trial endpoints"),
]

cal_texts = sorted({t for pair in PARAPHRASE_PAIRS + NON_PARAPHRASE_PAIRS for t in pair})
print(f"Encoding {len(cal_texts)} calibration texts...")
cal_emb = embed_texts(cal_texts)
cal_idx = {t: i for i, t in enumerate(cal_texts)}
cal_sim = cal_emb @ cal_emb.T

para_sims = [cal_sim[cal_idx[a], cal_idx[b]] for a, b in PARAPHRASE_PAIRS]
nonpara_sims = [cal_sim[cal_idx[a], cal_idx[b]] for a, b in NON_PARAPHRASE_PAIRS]
print(f"  paraphrase     : mean={np.mean(para_sims):.3f} range=[{np.min(para_sims):.3f},{np.max(para_sims):.3f}]")
print(f"  non-paraphrase : mean={np.mean(nonpara_sims):.3f} range=[{np.min(nonpara_sims):.3f},{np.max(nonpara_sims):.3f}]")

cal_results = []
for tau in np.arange(0.0, 1.05, 0.05):
    sens = np.mean([s >= tau for s in para_sims])
    spec = np.mean([s < tau for s in nonpara_sims])
    cal_results.append({"tau": tau, "sensitivity": sens, "specificity": spec,
                        "youden_j": sens + spec - 1})
cal_df = pd.DataFrame(cal_results)
best = cal_df.loc[cal_df["youden_j"].idxmax()]
TAU_STAR = float(best["tau"])
cal_df.to_csv(OUTPUT_DIR / "calibration_tau_sweep.csv", index=False)

print(f"\n  ★ τ* = {TAU_STAR:.2f}  (sensitivity={best['sensitivity']:.2f}, specificity={best['specificity']:.2f})")

Encoding 43 calibration texts...
  embed batch 1/1: 43 texts... 

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


done (1.5s)
  paraphrase     : mean=0.642 range=[0.451,0.863]
  non-paraphrase : mean=0.245 range=[0.125,0.355]

  ★ τ* = 0.40  (sensitivity=1.00, specificity=1.00)


## 9. Compute PlanScore for every generated plan

In [ ]:
# Collect every unique step (ground-truth + generated) and embed once.
all_steps = set()
for _, row in long_df.iterrows():
    all_steps.update(split_steps(row["ground_truth"]))
    all_steps.update(split_steps(row["generated"]))
step_list = sorted(all_steps)
step_to_idx = {s: i for i, s in enumerate(step_list)}
print(f"Unique steps to embed: {len(step_list)}")

step_emb = embed_texts(step_list)
np.save(OUTPUT_DIR / "step_embeddings.npy", step_emb)
with open(OUTPUT_DIR / "step_list.json", "w") as f:
    json.dump(step_list, f)

sim_matrix = step_emb @ step_emb.T
np.fill_diagonal(sim_matrix, 1.0)

score_rows = []
for _, row in long_df.iterrows():
    res = compute_plan_score(
        split_steps(row["ground_truth"]), split_steps(row["generated"]),
        sim_matrix, step_to_idx, TAU_STAR,
    )
    score_rows.append({"run": row["run"], "task_id": row["task_id"],
                       "config": row["config"], "use_sop": row["use_sop"],
                       "use_lit": row["use_lit"], "use_episodic": row["use_episodic"],
                       "tau": TAU_STAR, **res})

scores_df = pd.DataFrame(score_rows)
scores_df.to_csv(OUTPUT_DIR / "planning_eval_scores.csv", index=False)

print("\nMean PlanScore by config:")
for c in CONFIG_ORDER:
    print(f"  {CONFIG_LABELS[c]:>8}: {scores_df.loc[scores_df.config == c, 'plan_score'].mean():.3f}")
scores_df.head()

## 10. Summary table (mean ± 95% CI)

In [ ]:
def ci95(x):
    x = np.asarray(x, dtype=float)
    n = len(x)
    if n < 2:
        return 0.0
    return stats.sem(x) * stats.t.ppf(0.975, df=n - 1)


summary_df = (
    scores_df.groupby("config", as_index=False)
    .agg(
        n=("plan_score", "size"),
        plan_score=("plan_score", "mean"),
        plan_score_ci95=("plan_score", ci95),
        content_f1=("content_f1", "mean"),
        order_preservation=("order_preservation", "mean"),
        precision=("precision", "mean"),
        recall=("recall", "mean"),
    )
)
summary_df["config"] = pd.Categorical(summary_df["config"], categories=CONFIG_ORDER, ordered=True)
summary_df = summary_df.sort_values("config").reset_index(drop=True)
summary_df.to_csv(OUTPUT_DIR / "planning_eval_summary.csv", index=False)
summary_df

## 11. Statistical analysis (nonparametric)

Kruskal–Wallis across the four arms, then pairwise Mann–Whitney U with Holm
correction — the same nonparametric protocol as the ablation study, applied to the
deterministic `plan_score`.

In [ ]:
def nonparametric_config_tests(df, group_col="config", metric_col="plan_score", p_adjust="holm"):
    """Kruskal–Wallis omnibus + pairwise Mann–Whitney U with multiple-testing correction."""
    groups = [g for g in CONFIG_ORDER if g in set(df[group_col])]
    grouped = [df.loc[df[group_col] == g, metric_col].dropna().values for g in groups]

    _, kw_p = kruskal(*grouped)
    kw_df = pd.DataFrame([{"metric": metric_col, "n_groups": len(groups), "kruskal_p": float(kw_p)}])

    pairs, pvals = [], []
    for g1, g2 in combinations(groups, 2):
        x = df.loc[df[group_col] == g1, metric_col].dropna().values
        y = df.loc[df[group_col] == g2, metric_col].dropna().values
        _, p = mannwhitneyu(x, y, alternative="two-sided")
        pairs.append((g1, g2, len(x), len(y), float(np.median(x)), float(np.median(y))))
        pvals.append(p)

    pvals = np.asarray(pvals, dtype=float)
    reject, p_adj, _, _ = multipletests(pvals, method=p_adjust)
    pairwise_df = pd.DataFrame(
        [{"config_a": g1, "config_b": g2, "n_a": n1, "n_b": n2,
          "median_a": m1, "median_b": m2, "median_diff_a_minus_b": m1 - m2,
          "p_raw": float(pr), "p_adj": float(pa), "significant": bool(rj), "p_adjust": p_adjust}
         for (g1, g2, n1, n2, m1, m2), pr, pa, rj in zip(pairs, pvals, p_adj, reject)]
    ).sort_values(["p_adj", "p_raw"]).reset_index(drop=True)
    return kw_df, pairwise_df


kruskal_df, pairwise_df = nonparametric_config_tests(scores_df, metric_col="plan_score")
kruskal_df.to_csv(OUTPUT_DIR / "kruskal_planscore.csv", index=False)
pairwise_df.to_csv(OUTPUT_DIR / "pairwise_planscore.csv", index=False)
print(kruskal_df.to_string(index=False))
pairwise_df

## 12. Metric distributions per configuration

In [ ]:
metrics = ["precision", "recall", "content_f1", "order_preservation", "plan_score"]
metric_labels = {
    "precision": "Precision (0-1)",
    "recall": "Recall (0-1)",
    "content_f1": "Content F1 (0-1)",
    "order_preservation": "Order preservation (0-1)",
    "plan_score": "Plan score (0-1)",
}

W_MM, H_MM = 183, 115
fig, axes = plt.subplots(2, 3, figsize=(W_MM / 25.4, H_MM / 25.4))
fig.subplots_adjust(left=0.08, right=0.975, top=0.95, bottom=0.09, hspace=0.55, wspace=0.45)

for ax, metric, panel in zip(axes.flat, metrics, "ABCDE"):
    sns.boxplot(
        data=scores_df, x="config", y=metric, order=CONFIG_ORDER, hue="config",
        palette=CONFIG_COLORS, saturation=1.0, width=0.62, linewidth=0.75, fliersize=0,
        boxprops=dict(alpha=0.65), medianprops=dict(color="#222222", linewidth=1.0),
        whiskerprops=dict(linewidth=0.75), capprops=dict(linewidth=0.75), legend=False, ax=ax,
    )
    sns.stripplot(
        data=scores_df, x="config", y=metric, order=CONFIG_ORDER, hue="config",
        palette=CONFIG_COLORS, size=2.2, alpha=0.85, edgecolor="white", linewidth=0.3,
        jitter=0.18, legend=False, ax=ax,
    )
    ax.set_xticklabels([CONFIG_LABELS[c] for c in CONFIG_ORDER], rotation=20, ha="right")
    ax.set_xlabel("")
    ax.set_ylabel(metric_labels[metric], labelpad=2)
    ax.set_ylim(0, 1)
    ax.text(-0.18, 1.05, panel, transform=ax.transAxes, fontsize=9, fontweight="bold",
            va="bottom", ha="left")

axes.flat[-1].set_visible(False)
fig.savefig(OUTPUT_DIR / "fig_metric_distributions.svg")
fig.savefig(OUTPUT_DIR / "fig_metric_distributions.png", dpi=300)
plt.show()

## 13. Mean PlanScore by configuration (95% CI)

In [ ]:
fig, ax = plt.subplots(figsize=(4.2, 3.4))
ordered = summary_df.sort_values("plan_score", ascending=False)
ax.bar(
    [CONFIG_LABELS[c] for c in ordered["config"]],
    ordered["plan_score"], yerr=ordered["plan_score_ci95"], capsize=4,
    color=[CONFIG_COLORS[c] for c in ordered["config"]], edgecolor="black", linewidth=0.6,
)
ax.set_ylabel("Plan score (OP × F1)")
ax.set_xlabel("Configuration")
ax.set_title("Mean deterministic PlanScore by configuration (95% CI)",
             fontsize=9, fontweight="bold", loc="left", pad=8)
ax.set_ylim(0, max(ordered["plan_score"] + ordered["plan_score_ci95"]) * 1.15)
sns.despine()
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "fig_planscore_bars.svg")
fig.savefig(OUTPUT_DIR / "fig_planscore_bars.png", dpi=300)
plt.show()

## 14. Pairwise significance heatmap (Holm-adjusted)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap


def p_to_stars(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"


order = summary_df.sort_values("plan_score", ascending=False)["config"].tolist()
n = len(order)
pos = {c: i for i, c in enumerate(order)}
p_mat = np.full((n, n), np.nan)
for row in pairwise_df.itertuples(index=False):
    i, j = pos[row.config_a], pos[row.config_b]
    p_mat[i, j] = p_mat[j, i] = row.p_adj

with np.errstate(divide="ignore"):
    log_p = -np.log10(p_mat)
lower = np.where(np.tril(np.ones((n, n), dtype=bool), k=-1), log_p, np.nan)

cmap = LinearSegmentedColormap.from_list(
    "pval", ["#F7F7F7"] + sns.color_palette("crest", n_colors=5).as_hex(), N=256)
cmap.set_bad("white")

fig, ax = plt.subplots(figsize=(6.8, 5.4))
vmax = np.nanmax(log_p) * 1.05 if np.isfinite(np.nanmax(log_p)) else 1.0
im = ax.imshow(lower, cmap=cmap, aspect="auto", vmin=0, vmax=vmax, interpolation="nearest")

for i in range(n):
    for j in range(n):
        if j >= i or np.isnan(p_mat[i, j]):
            continue
        txt = p_to_stars(p_mat[i, j])
        color = "white" if log_p[i, j] > vmax * 0.55 else "#1a1a1a"
        ax.text(j, i, txt, ha="center", va="center", fontsize=11, color=color,
                fontweight="bold" if txt != "ns" else "normal")

ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels([CONFIG_LABELS[c] for c in order], rotation=30, ha="right")
ax.set_yticklabels([CONFIG_LABELS[c] for c in order])
ax.set_title("Pairwise Holm-adjusted p-values (−log10)", fontsize=9, fontweight="bold", loc="left", pad=8)
cbar = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03)
cbar.set_label("−log10(p_adj)")
ax.grid(False)
sns.despine(left=True, bottom=True)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "fig_pairwise_heatmap.svg")
fig.savefig(OUTPUT_DIR / "fig_pairwise_heatmap.png", dpi=300)
plt.show()